# EvolveGCN-O on Elliptic++ Actors

This notebook runs the **repo-faithful trainer** for **5 different random seeds** on Google Colab, saves each run under `models/`, and writes seed-level summaries under `results/`.

**What it does**
- mounts Google Drive
- prepares the repo inside Colab
- links `data/` to `MyDrive/data`
- runs **5 seeds**
- keeps **per-seed** `model.pt`, `metrics.json`, `config.json` under `models/`
- writes **summary statistics** over seeds under `results/seed_summaries/evolvegcn_o_ellipticpp_actors_5seeds/`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_SOURCE = "github"  # "github" or "drive"
REPO_URL = "https://github.com/koshimbetovv/gnn-robustness-blockchain-research.git"

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/gnn-robustness-blockchain"
DRIVE_DATA_DIR = "/content/drive/MyDrive/data"
WORKDIR = "/content/gnn-robustness-blockchain"

SEEDS = [42, 43, 44, 45, 46]
FORCE_FRESH_CLONE = True
SUMMARY_SUBDIR = "seed_summaries"

In [ ]:
import os
import re
import sys
import subprocess

def pip_install(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *args]
    print("Installing:", " ".join(cmd))
    subprocess.run(cmd, check=True)

pip_install("-U", "pip", "setuptools", "wheel")

import torch

torch_version_match = re.match(r"^\d+\.\d+\.\d+", torch.__version__)
torch_version = torch_version_match.group(0) if torch_version_match else torch.__version__.split("+")[0]
cuda_version = torch.version.cuda
cuda_tag = f"cu{cuda_version.replace('.', '')}" if cuda_version else "cpu"
wheel_url = f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"

extension_packages = ["pyg_lib", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv"]
for pkg in extension_packages:
    try:
        pip_install(pkg, "-f", wheel_url)
    except subprocess.CalledProcessError as exc:
        print(f"Warning: could not install {pkg} from {wheel_url}. Continuing. Details: {exc}")

pip_install("torch_geometric")
pip_install(
    "torch-geometric-temporal",
    "PyWavelets",
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "tqdm",
    "pyyaml",
    "networkx",
    "matplotlib",
    "xgboost",
)
print("Installation finished.")

In [ ]:
import os
import shlex
import shutil
import subprocess
from pathlib import Path

WORKDIR_PATH = Path(WORKDIR)
DRIVE_PROJECT_PATH = Path(DRIVE_PROJECT_DIR)
DRIVE_DATA_PATH = Path(DRIVE_DATA_DIR)

DRIVE_PROJECT_PATH.mkdir(parents=True, exist_ok=True)
DRIVE_DATA_PATH.mkdir(parents=True, exist_ok=True)

if REPO_SOURCE == "github":
    if FORCE_FRESH_CLONE and WORKDIR_PATH.exists():
        shutil.rmtree(WORKDIR_PATH)
    if not WORKDIR_PATH.exists():
        subprocess.run(["git", "clone", REPO_URL, str(WORKDIR_PATH)], check=True)
elif REPO_SOURCE == "drive":
    if WORKDIR_PATH.exists():
        shutil.rmtree(WORKDIR_PATH)
    subprocess.run(
        [
            "bash",
            "-lc",
            f'rsync -a "{DRIVE_PROJECT_PATH.as_posix().rstrip("/")}/" "{WORKDIR_PATH.as_posix().rstrip("/")}/"'
        ],
        check=True,
    )
else:
    raise ValueError("REPO_SOURCE must be either 'github' or 'drive'.")

for folder_name in ["models", "results"]:
    target = DRIVE_PROJECT_PATH / folder_name
    target.mkdir(parents=True, exist_ok=True)

    link = WORKDIR_PATH / folder_name
    if link.is_symlink() or link.exists():
        if link.is_symlink() or link.is_file():
            link.unlink()
        else:
            shutil.rmtree(link)
    os.symlink(target, link)

data_link = WORKDIR_PATH / "data"
if data_link.is_symlink() or data_link.exists():
    if data_link.is_symlink() or data_link.is_file():
        data_link.unlink()
    else:
        shutil.rmtree(data_link)
os.symlink(DRIVE_DATA_PATH, data_link)

os.chdir(WORKDIR_PATH)
print("Working directory:", WORKDIR_PATH)
print("Data symlink      :", data_link, "->", DRIVE_DATA_PATH)
print("Models symlink    :", WORKDIR_PATH / "models", "->", DRIVE_PROJECT_PATH / "models")
print("Results symlink   :", WORKDIR_PATH / "results", "->", DRIVE_PROJECT_PATH / "results")

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd

REQUESTED_METRICS = [
    "f1_pos",
    "precision_pos",
    "recall_pos",
    "f1_macro",
    "precision_macro",
    "recall_macro",
    "accuracy",
    "training_time",
]

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

def get_new_run_dir(models_dir, before_dirs):
    models_dir = Path(models_dir)
    after_dirs = {p.resolve() for p in models_dir.glob("*") if p.is_dir()}
    new_dirs = sorted(after_dirs - before_dirs, key=lambda p: p.stat().st_mtime)
    if new_dirs:
        return new_dirs[-1]
    all_dirs = sorted(after_dirs, key=lambda p: p.stat().st_mtime)
    if not all_dirs:
        raise RuntimeError(f"No run directory found under {models_dir}")
    return all_dirs[-1]

def normalize_metrics_payload(payload, trainer_kind, training_time, seed, run_dir):
    run_dir = str(run_dir)

    if trainer_kind == "cosemignn":
        src = payload.get("test", payload["summary_concat_all_test_predictions"])
        precision_neg = float(src["precision_negative"])
        precision_pos = float(src["precision_positive"])
        recall_neg = float(src["recall_negative"])
        recall_pos = float(src["recall_positive"])

        normalized = {
            "f1_pos": float(src["f1_positive"]),
            "precision_pos": precision_pos,
            "recall_pos": recall_pos,
            "f1_macro": float(src["f1_macro"]),
            "precision_macro": float(src.get("precision_macro", (precision_neg + precision_pos) / 2.0)),
            "recall_macro": float(src.get("recall_macro", (recall_neg + recall_pos) / 2.0)),
            "accuracy": float(src.get("accuracy", src.get("acc"))),
            "training_time": float(training_time),
        }
    else:
        src = payload["test"]
        normalized = {
            "f1_pos": float(src["f1_pos"]),
            "precision_pos": float(src["precision_pos"]),
            "recall_pos": float(src["recall_pos"]),
            "f1_macro": float(src["f1_macro"]),
            "precision_macro": float(src["precision_macro"]),
            "recall_macro": float(src["recall_macro"]),
            "accuracy": float(src.get("accuracy", src.get("acc"))),
            "training_time": float(training_time),
        }

    normalized["seed"] = int(seed)
    normalized["run_dir"] = run_dir
    return normalized

def patch_metrics_file(run_dir, trainer_kind, normalized_metrics):
    run_dir = Path(run_dir)
    metrics_path = run_dir / "metrics.json"
    payload = json.loads(metrics_path.read_text())

    payload["test"] = dict(payload.get("test", {}))
    payload["test"].update({
        "f1_pos": float(normalized_metrics["f1_pos"]),
        "precision_pos": float(normalized_metrics["precision_pos"]),
        "recall_pos": float(normalized_metrics["recall_pos"]),
        "f1_macro": float(normalized_metrics["f1_macro"]),
        "precision_macro": float(normalized_metrics["precision_macro"]),
        "recall_macro": float(normalized_metrics["recall_macro"]),
        "accuracy": float(normalized_metrics["accuracy"]),
        "training_time": float(normalized_metrics["training_time"]),
        "seed": int(normalized_metrics["seed"]),
    })
    if "acc" in payload["test"]:
        payload["test"]["acc"] = float(payload["test"]["accuracy"])

    payload["colab_multiseed"] = {
        "trainer_kind": trainer_kind,
        "seed": int(normalized_metrics["seed"]),
        "run_dir": str(run_dir),
        "requested_metrics": REQUESTED_METRICS,
    }
    write_json(metrics_path, payload)

def save_summary_outputs(records, summary_dir):
    summary_dir = Path(summary_dir)
    summary_dir.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(records)
    metric_df = df[["seed", "run_dir", *REQUESTED_METRICS]].copy()
    metric_df.to_csv(summary_dir / "per_seed_metrics.csv", index=False)
    write_json(summary_dir / "per_seed_metrics.json", metric_df.to_dict(orient="records"))

    rows = []
    for metric in REQUESTED_METRICS:
        values = metric_df[metric].astype(float)
        rows.append({
            "metric": metric,
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
            "min": float(values.min()),
            "max": float(values.max()),
        })
    summary_df = pd.DataFrame(rows)
    summary_df.to_csv(summary_dir / "summary_statistics.csv", index=False)
    write_json(summary_dir / "summary_statistics.json", summary_df.to_dict(orient="records"))

    return metric_df, summary_df

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

sys.path.insert(0, str(WORKDIR_PATH))

from scripts.training.train_evolvegcn_utils import (
    DualAdam,
    WeightedCrossEntropy,
    build_evolvegcn_model,
    evaluate_evolvegcn,
    move_sample,
    resolve_class_weights,
)
from scripts.training.train_paper_utils import get_device
from src.utils.logging import print_eval_epoch_metrics, print_final_metrics, save_run
from src.utils.seed import seed_from_config
from src.datasets.evolvegcn_ellipticpp_actors import EvolveGCNActorsConfig, EvolveGCNEllipticPPActorsDataset

CONFIG_PATH = Path(WORKDIR_PATH / "config/models/train_evolvegcn_ellipticpp_actors_o.json")
TRAINER_KIND = "evolvegcn"
SUMMARY_DIR = WORKDIR_PATH / "results" / SUMMARY_SUBDIR / "evolvegcn_o_ellipticpp_actors_5seeds"

original_config_text = CONFIG_PATH.read_text(encoding="utf-8")

def run_once(seed):
    cfg = json.loads(original_config_text)
    cfg["seed"] = int(seed)
    cfg.setdefault("save", {})
    base_prefix = cfg["save"].get("prefix") or Path("config/models/train_evolvegcn_ellipticpp_actors_o.json").stem.replace("train_", "")
    cfg["save"]["prefix"] = f"{base_prefix}_seed{seed}"
    cfg["save"]["save_dir"] = "models"

    seed_from_config(cfg)
    device = get_device(allow_mps=False)

    dataset = EvolveGCNEllipticPPActorsDataset(EvolveGCNActorsConfig(**cfg["data"]))
    sequence = dataset.get_sequence()

    model = build_evolvegcn_model(sequence.num_features, cfg).to(device)
    class_weights = resolve_class_weights(sequence.train_samples, cfg)
    loss_fn = WeightedCrossEntropy(class_weights, device=device)
    optim = DualAdam(model, lr=cfg["training"]["lr"])

    print(f"Class weights: {class_weights}")
    print(
        f"\n=== Training EvolveGCN-O on Elliptic++ Actors ===\n"
        f"Train windows: {len(sequence.train_samples)} | Test windows: {len(sequence.test_samples)} | "
        f"num_nodes={sequence.num_nodes} | input_dim={sequence.num_features}"
    )

    start_time = time.time()
    log_every = cfg["training"].get("log_every", 20)
    grad_acc = max(int(cfg["training"].get("steps_accum_gradients", 1)), 1)

    for epoch in range(1, cfg["training"]["epochs"] + 1):
        model.train()
        optim.zero_grad(set_to_none=True)

        for step, sample in enumerate(sequence.train_samples, start=1):
            hist_adj_list, hist_ndFeats_list, node_mask_list, label_idx, label_vals = move_sample(sample, device)
            logits = model(hist_adj_list, hist_ndFeats_list, node_mask_list, label_idx)
            loss = loss_fn(logits, label_vals) / grad_acc
            loss.backward()

            if step % grad_acc == 0 or step == len(sequence.train_samples):
                optim.step()
                optim.zero_grad(set_to_none=True)

        if epoch == 1 or epoch % log_every == 0 or epoch == cfg["training"]["epochs"]:
            train_metrics = evaluate_evolvegcn(model, sequence.train_samples, loss_fn, device)
            test_metrics = evaluate_evolvegcn(model, sequence.test_samples, loss_fn, device)
            print_eval_epoch_metrics(epoch, train_metrics, test_metrics)

    test_metrics = evaluate_evolvegcn(model, sequence.test_samples, loss_fn, device)
    print_final_metrics("EvolveGCN-O on Elliptic++ Actors", test_metrics)
    run_dir = Path(save_run(model, test_metrics, cfg, cfg["save"]["prefix"]))
    elapsed = time.time() - start_time

    payload = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    normalized = normalize_metrics_payload(
        payload=payload,
        trainer_kind=TRAINER_KIND,
        training_time=elapsed,
        seed=seed,
        run_dir=run_dir,
    )
    patch_metrics_file(run_dir, TRAINER_KIND, normalized)
    return normalized

records = []
try:
    for seed in SEEDS:
        print("\n" + "=" * 80)
        print(f"Running EvolveGCN-O on Elliptic++ Actors | seed={seed}")
        print("=" * 80)
        records.append(run_once(seed))

    per_seed_df, summary_df = save_summary_outputs(records, SUMMARY_DIR)
    display(per_seed_df)
    display(summary_df)
    print(f"Saved summary files to: {SUMMARY_DIR}")
finally:
    CONFIG_PATH.write_text(original_config_text, encoding="utf-8")